# 04 · Simple spectral extraction

So far we've gone *forward*: source spectrum → dispersed image. This notebook goes *backward* — pull a 1D spectrum back out of a dispersed image. `roman_disperser` has no extraction routine, so we build a minimal **boxcar** extractor from the optical model itself, in three steps: (1) the **trace** maps wavelength → detector pixel; (2) **autodiff** of the trace gives the dispersion dλ/pixel; (3) sum the cross-dispersion pixels at each wavelength and rescale. We check it against the known input spectrum, then package it for reuse.

This is a teaching extractor (boxcar, single source, no contamination handling) — enough to recover a clean spectrum and to drive the line-profile lesson in notebook 05.

## 0 · Setup — make a dispersed star to extract

In [ ]:
import os
from pathlib import Path
os.environ.setdefault("JAX_COMPILATION_CACHE_DIR", str(Path.home() / ".cache" / "roman_grs_jax"))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import AsinhNorm
import jax
import jax.numpy as jnp

from roman_disperser import paths, psf_model, star_disperser
from roman_disperser.optical_model import RomanOpticalModel
import roman_disperser.optical_model_jax as omj
import tutorial_helpers as th

SCA = 5
model = RomanOpticalModel(config_file=str(paths.optical_model_path()))
opt = omj.make_sca_payload(model, sca=SCA, order="1")
psf = psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order="1",
                                        cache_dir=str(paths.psf_cache_dir()), verbose=False)
star_disp = star_disperser.make_star_disperser(psf, opt)

wl_um, _, _ = th.grism_wavelength_grid()
X_STAR, Y_STAR = 2000.0, 2000.0
_, counts_in = th.template_to_counts("g0v", 16.0, sca=SCA, order="1", wl_um=wl_um)
img = np.asarray(star_disp(X_STAR, Y_STAR, jnp.asarray(wl_um),
                           jnp.asarray(counts_in), jnp.zeros((4088, 4088), jnp.float32)))
print(f"dispersed a G0V star (the 'observation' we'll extract); {wl_um.size} wavelengths")

## 1 · The spectral trace

The optical model tells us, for an undispersed source at `(x, y)`, where light of each wavelength lands. We chain the JAX transforms `sca_to_fpa → trace_beam → mpa_to_sca` to get the **trace**: pixel position as a function of wavelength. Overlaid on the dispersed image, it follows the spectrum exactly. (Dispersion runs along **y**.)

In [ ]:
def spectral_trace(payload, xsca, ysca, wl_um):
    wl = jnp.asarray(wl_um)
    xfpa, yfpa = omj.sca_to_fpa(payload, xsca, ysca)
    xmpa, ympa = omj.trace_beam(payload, jnp.broadcast_to(xfpa, wl.shape),
                                jnp.broadcast_to(yfpa, wl.shape), wl)
    tx, ty = omj.mpa_to_sca(payload, xmpa, ympa)
    return np.asarray(tx).ravel(), np.asarray(ty).ravel()

trace_x, trace_y = spectral_trace(opt, X_STAR, Y_STAR, wl_um)

ys, xs = np.nonzero(img)
pad = 15
bx0, bx1, by0, by1 = xs.min()-pad, xs.max()+pad, ys.min()-pad, ys.max()+pad
fig, ax = plt.subplots(figsize=(4, 7))
ax.imshow(img[by0:by1, bx0:bx1], origin="lower", cmap="inferno", extent=[bx0, bx1, by0, by1],
          norm=AsinhNorm(linear_width=img.max()*0.002, vmin=0, vmax=img.max()))
ax.plot(trace_x - 1, trace_y - 1, color="cyan", lw=0.8, label="optical-model trace")
ax.legend(loc="upper right", fontsize=8)
ax.set(title="dispersed star + trace", xlabel="x [pix]", ylabel="y [pix]")
fig.tight_layout()

## 2 · The dispersion, by autodiff

To turn "counts per pixel-row" into "counts per wavelength bin" we need the local dispersion **dy/dλ** (pixels per micron). Because the trace is a differentiable JAX function, we get it exactly with `jax.grad` — no finite differences — and `vmap` it over the wavelength grid.

In [ ]:
def trace_y_only(payload, xsca, ysca, wl):
    wl1 = jnp.atleast_1d(wl)
    xfpa, yfpa = omj.sca_to_fpa(payload, xsca, ysca)
    xmpa, ympa = omj.trace_beam(payload, jnp.broadcast_to(xfpa, wl1.shape),
                                jnp.broadcast_to(yfpa, wl1.shape), wl1)
    _, ty = omj.mpa_to_sca(payload, xmpa, ympa)
    return ty[0]

dydl_fn = jax.jit(jax.vmap(jax.grad(lambda w: trace_y_only(opt, X_STAR, Y_STAR, w))))
dy_dlam = np.asarray(dydl_fn(jnp.asarray(wl_um)))     # pixels per micron

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wl_um, np.abs(dy_dlam), color="C2")
ax.set(xlabel="wavelength [µm]", ylabel="|dy/dλ| [pix/µm]", title="dispersion along the trace")
fig.tight_layout()
print(f"~{np.abs(dy_dlam).mean():.0f} pix/µm  ≈  {1e4/np.abs(dy_dlam).mean():.1f} Å per pixel")

## 3 · Boxcar extraction

At each wavelength, sum the pixels within ±`aperture` of the trace in the cross-dispersion (x) direction, then multiply by `|dy/dλ|·Δλ` to convert to a count rate per wavelength bin. Compared against the spectrum we fed in, the recovery is excellent.

In [ ]:
dlam_um = wl_um[1] - wl_um[0]
aperture = 12
extracted = np.zeros(len(wl_um))
for i in range(len(wl_um)):
    ix = int(round(trace_x[i])) - 1       # 1-indexed FITS -> 0-indexed array
    iy = int(round(trace_y[i])) - 1
    if 0 <= ix < img.shape[1] and 0 <= iy < img.shape[0]:
        extracted[i] = img[iy, max(0, ix-aperture):ix+aperture+1].sum()
extracted = extracted * np.abs(dy_dlam) * dlam_um

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(wl_um, counts_in, color="C0", lw=1.3, label="input spectrum")
ax.plot(wl_um, extracted, color="C3", lw=0.8, label=f"extracted (±{aperture} pix)")
ax.set(xlabel="wavelength [µm]", ylabel="count rate [e⁻/s per bin]",
       title="extracted vs input")
ax.legend(); fig.tight_layout()

m = (extracted > 0) & (counts_in > 0)
print(f"recovered {np.median(extracted[m]/counts_in[m])*100:.0f}% of the flux at ±{aperture} pix")

## 4 · Aperture, and what's left out

A boxcar trades completeness for simplicity: a wider aperture captures more of the PSF wings (more flux) at the cost of more background and contamination. The library has no extractor, so the function above is packaged as `tutorial_helpers.extract_1d` for the next notebook.

In [ ]:
for ap in [4, 8, 12, 20]:
    e = th.extract_1d(img, opt, X_STAR, Y_STAR, wl_um, aperture=ap)
    m = (e > 0) & (counts_in > 0)
    print(f"aperture ±{ap:2d} pix → {np.median(e[m]/counts_in[m])*100:.0f}% of the flux")

## 5 · The full scene, and zeroth-order contamination

Every source also makes a **0th order** — a compact, almost-undispersed image — that lands well away from its 1st-order spectrum. A *bright* star's 0th order can fall right on a *faint* galaxy's 1st-order trace and masquerade as an emission line. Because the 0th order moves with the telescope **roll** (notebook 03) while a real line does not, re-observing at a different roll tells them apart.

To see the whole picture, we disperse a bright star and a fainter galaxy (with one real emission line) through **both** orders 0 and 1, and place the star so its 0th order lands on the galaxy's 1st-order trace just beside the line.

In [ ]:
from roman_disperser import galaxy_disperser, sersic

ORD = ["0", "1"]
opt_o = {o: omj.make_sca_payload(model, sca=SCA, order=o) for o in ORD}
psf_o = {o: psf_model.get_or_make_psf_payload(detector=f"WFI{SCA:02d}", order=o,
             cache_dir=str(paths.psf_cache_dir()), verbose=False) for o in ORD}
star_disp_o = {o: star_disperser.make_star_disperser(psf_o[o], opt_o[o]) for o in ORD}
gal_disp_o  = {o: galaxy_disperser.make_galaxy_disperser(psf_o[o], opt_o[o]) for o in ORD}
OV = psf_o["1"]["oversample"]
CX = CY = 2044.0

def roll_xy(x, y, deg):                     # rotate about the SCA centre (as in notebook 03)
    t = np.deg2rad(deg); c, s = np.cos(t), np.sin(t)
    return CX + c*(x-CX) - s*(y-CY), CY + s*(x-CX) + c*(y-CY)

# a FAINT galaxy: redshifted continuum + one bright emission line at 1.5 µm
GX, GY, LINE = 2100.0, 2100.0, 1.5
gal_counts = {
    "1": jnp.asarray(th.template_to_counts("kc96_starb1", 21.0, sca=SCA, order="1",
                                           wl_um=wl_um, redshift=1.0)[1]
                     + 80.0*np.exp(-0.5*((wl_um - LINE)/0.0008)**2)),
    "0": jnp.asarray(th.template_to_counts("kc96_starb1", 21.0, sca=SCA, order="0",
                                           wl_um=wl_um, redshift=1.0)[1]),
}
gal_img = sersic.make_sersic_image(sersic.catalog_r_eff_to_pixels(0.3, 0.11, OV),
                                   n=1.0, ba=1.0, theta=0.0, npix=30*OV)
gal_img = jnp.asarray(gal_img / gal_img.sum())

# a BRIGHT star: its 0th order is the contaminant. (per-order counts use the
# real sensitivities, so the 0th order is ~0.3% of the 1st — only a bright star
# contaminates.)
star_counts = {o: jnp.asarray(th.template_to_counts("g0v", 13.5, sca=SCA, order=o, wl_um=wl_um)[1])
               for o in ORD}

# place the star so its 0th order lands ~50 px (~500 Å) from the galaxy's line.
line_x, line_y = [float(v[0]) for v in th.spectral_trace(opt_o["1"], GX, GY, [LINE])]
def order0_centroid(x, y):
    im = np.asarray(star_disp_o["0"](x, y, jnp.asarray(wl_um), jnp.ones_like(jnp.asarray(wl_um))*8.0,
                                     jnp.zeros((4088, 4088), jnp.float32)))
    yy, xx = np.nonzero(im); w = im[yy, xx]
    return np.sum(xx*w)/w.sum() + 1, np.sum(yy*w)/w.sum() + 1
SX, SY = line_x, line_y + 50
for _ in range(6):
    cx, cy = order0_centroid(SX, SY); SX += line_x - cx; SY += (line_y + 50) - cy
print(f"galaxy at ({GX:.0f}, {GY:.0f}); bright star at ({SX:.0f}, {SY:.0f})")

Disperse the pair through both orders. The full SCA shows all four pieces: the star's bright 1st order, the galaxy's 1st order with its line, the galaxy's faint 0th order, and — sitting right on the galaxy's trace — the star's 0th order.

In [ ]:
def disperse_scene(deg):
    gx, gy = roll_xy(GX, GY, deg)
    sx, sy = roll_xy(SX, SY, deg)
    Z = lambda: jnp.zeros((4088, 4088), jnp.float32)
    parts = {
        "galaxy 1st (+line)":     np.asarray(gal_disp_o["1"](gal_img, gx, gy, gal_counts["1"], jnp.asarray(wl_um), Z())),
        "galaxy 0th":             np.asarray(gal_disp_o["0"](gal_img, gx, gy, gal_counts["0"], jnp.asarray(wl_um), Z())),
        "star 1st":               np.asarray(star_disp_o["1"](sx, sy, jnp.asarray(wl_um), star_counts["1"], Z())),
        "star 0th (contaminant)": np.asarray(star_disp_o["0"](sx, sy, jnp.asarray(wl_um), star_counts["0"], Z())),
    }
    return parts, (gx, gy)

def centroid(im):
    yy, xx = np.nonzero(im); w = im[yy, xx]
    return np.sum(xx*w)/w.sum(), np.sum(yy*w)/w.sum()

parts0, (gx0, gy0) = disperse_scene(0.0)
full0 = sum(parts0.values())

ys, xs = np.nonzero(full0)
bx0, bx1, by0, by1 = xs.min()-60, xs.max()+60, ys.min()-60, ys.max()+60
fig, ax = plt.subplots(figsize=(4.5, 8))
ax.imshow(full0[by0:by1, bx0:bx1], origin="lower", cmap="inferno", extent=[bx0, bx1, by0, by1],
          norm=AsinhNorm(linear_width=full0.max()*5e-5, vmin=0, vmax=full0.max()))
for name, im in parts0.items():
    cx, cy = centroid(im)
    ax.annotate(name, (cx, cy), (bx1+8, cy), color="cyan", fontsize=8, va="center",
                arrowprops=dict(color="cyan", arrowstyle="->", lw=0.6))
ax.set(title="full dispersed scene (roll 0°)", xlabel="x [pix]", ylabel="y [pix]", xticks=[])
fig.tight_layout()

Now extract the galaxy's 1st-order spectrum at two rolls. At roll 0 the star's 0th order shows up as a second "line" ~500 Å from the real one; at a different roll it has moved off the galaxy's trace while the real line is unchanged.

In [ ]:
prof0 = th.extract_1d(full0, opt_o["1"], gx0, gy0, wl_um, aperture=20)
partsR, (gxR, gyR) = disperse_scene(20.0)
profR = th.extract_1d(sum(partsR.values()), opt_o["1"], gxR, gyR, wl_um, aperture=20)

fig, ax = plt.subplots(figsize=(7, 3.6))
ax.plot(wl_um*1e4, prof0, color="C3", label="roll = 0°")
ax.plot(wl_um*1e4, profR, color="C0", label="roll = 20°")
ax.axvline(LINE*1e4, ls=":", color="0.5", lw=1, label="true line")
ax.set(xlabel="wavelength [Å]", ylabel="extracted counts [e⁻/s]", xlim=(14000, 16500),
       title="galaxy spectrum, two rolls: the real line stays, the contaminant moves")
ax.legend(); fig.tight_layout()

This is why a boxcar (or any) extraction is only as good as the scene around it, and why grism surveys observe at **multiple rolls**: a feature that moves with roll is a contaminant, not a line. The production pipeline tracks which sources have 0th orders overlapping each SCA so a contamination model can be built — but the cheap, robust check is a second roll.

## Recap

- The **trace** (`sca_to_fpa → trace_beam → mpa_to_sca`) maps wavelength → pixel; **`jax.grad`** of it gives the dispersion.
- A **boxcar** sum along the cross-dispersion direction, scaled by `|dy/dλ|·Δλ`, recovers the 1D spectrum.
- Packaged as `tutorial_helpers.extract_1d(image, payload, x, y, wl_um, aperture=...)`.
- A bright source's **0th order** can land on another's spectrum and fake a line; it moves with **roll**, so a second roll exposes it.

**Next — [05 · Position angle and emission-line profiles](05_pa_line_profiles.ipynb).** We disperse an *elongated* galaxy with an emission line and watch the extracted line width change with the galaxy's orientation — morphological broadening, a real systematic in slitless spectroscopy.